<a href="https://colab.research.google.com/github/Rohit-U76/RL/blob/main/RL_Assignment_No_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

# -------------------------------
# Environment Settings
# -------------------------------

num_states = 5
states = range(num_states)
actions = ["L", "R"]

gamma = 0.9
theta = 1e-4

# Rewards for states
rewards = [100, 0, 0, 0, 40]   # State 1 and State 5 are terminal


# -------------------------------
# Transition Function
# -------------------------------

def next_state(s, a):
    if a == "L":
        return max(s - 1, 0)
    else:
        return min(s + 1, num_states - 1)


# ===============================
# PART 1 : VALUE ITERATION
# ===============================

V = np.zeros(num_states)

while True:
    delta = 0

    for s in states:

        # Skip terminal states
        if s in [0, num_states - 1]:
            continue

        v = V[s]
        action_values = []

        for a in actions:
            s_next = next_state(s, a)
            r = rewards[s_next]

            action_values.append(r + gamma * V[s_next])

        V[s] = max(action_values)

        delta = max(delta, abs(v - V[s]))

    if delta < theta:
        break


print("VALUE ITERATION RESULTS\n")

for s in states:
    print(f"State {s+1} → V = {V[s]:.2f}")


# ===============================
# PART 2 : Q-LEARNING
# ===============================

alpha = 0.1
epsilon = 0.2
episodes = 50

# Initialize Q-table
Q = np.zeros((num_states, len(actions)))


def choose_action(state):

    # Exploration
    if random.uniform(0, 1) < epsilon:
        return random.choice([0, 1])

    # Exploitation
    else:
        return np.argmax(Q[state])


print("\nQ-LEARNING TRAINING\n")

for episode in range(episodes):

    state = 3   # Start at State 4
    print(f"Episode {episode+1}")

    while state not in [0, num_states - 1]:

        action_idx = choose_action(state)
        action = actions[action_idx]

        next_s = next_state(state, action)
        reward = rewards[next_s]

        # Q-learning update
        Q[state, action_idx] += alpha * (
            reward + gamma * np.max(Q[next_s]) - Q[state, action_idx]
        )

        V_state = np.max(Q[state])

        print(
            f"State {state+1} | "
            f"Q(L)={Q[state,0]:.2f}, "
            f"Q(R)={Q[state,1]:.2f}, "
            f"V={V_state:.2f}"
        )

        state = next_s


# ===============================
# PART 3 : BEST PATH EXTRACTION
# ===============================

state = 3  # Start position
path = [state + 1]
total_reward = 0

while state not in [0, num_states - 1]:

    action_idx = np.argmax(Q[state])

    state = next_state(state, actions[action_idx])

    path.append(state + 1)

    total_reward += rewards[state]


print("\nBEST PATH FOUND\n")

print(" → ".join(map(str, path)))

print("Total Reward:", total_reward)

VALUE ITERATION RESULTS

State 1 → V = 0.00
State 2 → V = 100.00
State 3 → V = 90.00
State 4 → V = 81.00
State 5 → V = 0.00

Q-LEARNING TRAINING

Episode 1
State 4 | Q(L)=0.00, Q(R)=0.00, V=0.00
State 3 | Q(L)=0.00, Q(R)=0.00, V=0.00
State 2 | Q(L)=10.00, Q(R)=0.00, V=10.00
Episode 2
State 4 | Q(L)=0.00, Q(R)=0.00, V=0.00
State 3 | Q(L)=0.90, Q(R)=0.00, V=0.90
State 2 | Q(L)=19.00, Q(R)=0.00, V=19.00
Episode 3
State 4 | Q(L)=0.08, Q(R)=0.00, V=0.08
State 3 | Q(L)=0.90, Q(R)=0.01, V=0.90
State 4 | Q(L)=0.15, Q(R)=0.00, V=0.15
State 3 | Q(L)=2.52, Q(R)=0.01, V=2.52
State 2 | Q(L)=27.10, Q(R)=0.00, V=27.10
Episode 4
State 4 | Q(L)=0.37, Q(R)=0.00, V=0.37
State 3 | Q(L)=4.71, Q(R)=0.01, V=4.71
State 2 | Q(L)=34.39, Q(R)=0.00, V=34.39
Episode 5
State 4 | Q(L)=0.75, Q(R)=0.00, V=0.75
State 3 | Q(L)=7.33, Q(R)=0.01, V=7.33
State 2 | Q(L)=40.95, Q(R)=0.00, V=40.95
Episode 6
State 4 | Q(L)=1.34, Q(R)=0.00, V=1.34
State 3 | Q(L)=10.28, Q(R)=0.01, V=10.28
State 2 | Q(L)=46.86, Q(R)=0.00, V=46.86


# Practical Assignment 3

# Reinforcement Learning: State Value (V), Action Value (Q), and Best Path

---

# 1. Introduction

This assignment demonstrates how a **rover agent** learns to reach a goal state using reinforcement learning concepts. The rover moves inside a **1-dimensional environment consisting of 5 states**.

The agent can perform two actions:

* Move **Left (L)**
* Move **Right (R)**

The rover must determine the **best path to reach a goal state** while maximizing the total reward. During the process, the rover calculates:

* **State Value (V)** — the expected reward from a state
* **Action Value (Q)** — the expected reward from taking an action in a state

Two reinforcement learning approaches are used:

1. **Value Iteration**
2. **Q-Learning with ε-Greedy exploration**

---

# 2. Objective

* To calculate **state values V(s)** using Value Iteration
* To calculate **action values Q(s,a)** using Q-Learning
* To observe learning updates at each step
* To determine the **optimal path that maximizes cumulative reward**

---

# 3. Problem Description

The rover environment consists of **5 states arranged in a line**.

```
[State1] [State2] [State3] [State4] [State5]
```

### Start Position

* Rover starts at **State 4**

### Goal States

* **State 1 → Reward = 100**
* **State 5 → Reward = 40**

### Other States

* **Reward = 0**

### Available Actions

| Action | Description |
| ------ | ----------- |
| L      | Move Left   |
| R      | Move Right  |

The rover must reach a goal state while maximizing reward.

---

# 4. Reinforcement Learning Components

### Agent

Rover

### Environment

1-Dimensional grid with 5 states

### States

```
S1 S2 S3 S4 S5
```

### Rewards

| State   | Reward |
| ------- | ------ |
| State 1 | 100    |
| State 2 | 0      |
| State 3 | 0      |
| State 4 | 0      |
| State 5 | 40     |

### Terminal States

* State 1
* State 5

---

# 5. Algorithms Used

## 5.1 Value Iteration

Value Iteration calculates the **optimal state value function** using the Bellman optimality equation:

```
V(s) = max [ R(s,a) + γ V(s') ]
```

Where:

* **V(s)** = value of state
* **R(s,a)** = reward received
* **γ** = discount factor
* **s'** = next state

---

## 5.2 Q-Learning

Q-Learning calculates **action values (Q-values)** for each state-action pair.

Update equation:

```
Q(s,a) = Q(s,a) + α [ R + γ max(Q(s',a')) − Q(s,a) ]
```

Where:

* **α** = learning rate
* **γ** = discount factor
* **ε** = exploration probability

The agent uses **ε-greedy exploration** to balance exploration and exploitation.

---

# 6. Python Implementation

## Part 1: Value Iteration

```python
import numpy as np

# Environment settings
num_states = 5
states = range(num_states)
actions = ["L", "R"]

gamma = 0.9
theta = 1e-4

# Rewards
rewards = [100, 0, 0, 0, 40]

# Transition function
def next_state(s, a):
    if a == "L":
        return max(s - 1, 0)
    else:
        return min(s + 1, num_states - 1)

# Initialize value function
V = np.zeros(num_states)

# Value Iteration
while True:
    delta = 0
    for s in states:
        if s in [0, num_states - 1]:
            continue

        v = V[s]
        action_values = []

        for a in actions:
            s_next = next_state(s, a)
            r = rewards[s_next]
            action_values.append(r + gamma * V[s_next])

        V[s] = max(action_values)
        delta = max(delta, abs(v - V[s]))

    if delta < theta:
        break

print("VALUE ITERATION RESULTS")

for s in states:
    print(f"State {s+1} → V = {V[s]:.2f}")
```

---

# Part 2: Q-Learning with ε-Greedy Exploration

```python
import random
import numpy as np

alpha = 0.1
gamma = 0.9
epsilon = 0.2
episodes = 50

Q = np.zeros((num_states, len(actions)))

def choose_action(state):
    if random.uniform(0,1) < epsilon:
        return random.choice([0,1])
    else:
        return np.argmax(Q[state])

for episode in range(episodes):

    state = 3
    print(f"\nEpisode {episode+1}")

    while state not in [0, num_states-1]:

        action_idx = choose_action(state)
        action = actions[action_idx]

        next_s = next_state(state, action)
        reward = rewards[next_s]

        Q[state, action_idx] += alpha * (
            reward + gamma * np.max(Q[next_s]) - Q[state, action_idx]
        )

        V_state = np.max(Q[state])

        print(
            f"State {state+1} | "
            f"Q(L)={Q[state,0]:.2f}, Q(R)={Q[state,1]:.2f}, "
            f"V={V_state:.2f}"
        )

        state = next_s
```

---

# Part 3: Best Path Extraction

```python
state = 3
path = [state + 1]
total_reward = 0

while state not in [0, num_states - 1]:

    action_idx = np.argmax(Q[state])
    state = next_state(state, actions[action_idx])

    path.append(state + 1)
    total_reward += rewards[state]

print("\nBEST PATH FOUND")

print(" → ".join(map(str, path)))

print("Total Reward:", total_reward)
```

---

# 7. Input

* No external input required
* The rover automatically chooses actions during learning

---

# 8. Output

The program displays:

* **State Values V(s)** from Value Iteration
* **Action Values Q(s,a)** during Q-Learning
* **Best path from the start state to a goal state**
* **Total reward collected**

Example output:

```
State 4 | Q(L)=18.00, Q(R)=7.20, V=18.00
State 3 | Q(L)=32.40, Q(R)=16.20, V=32.40
```

Best Path Example:

```
4 → 3 → 2 → 1
Total Reward: 100
```

---

# 9. Result and Observation

* The rover successfully learns the **optimal strategy** using reinforcement learning.
* The algorithm identifies **State 1 as the optimal goal** since it provides a higher reward (100).
* Over multiple episodes, the **Q-values converge** to the optimal policy.

---

# 10. Advantages

* Demonstrates reinforcement learning concepts clearly
* Shows both **model-based (Value Iteration)** and **model-free (Q-Learning)** approaches
* Helps understand **state values and action values**

---

# 11. Limitations

* Environment is very small
* Only two possible actions
* No complex state transitions

---

# 12. Conclusion

This assignment demonstrates how reinforcement learning algorithms can be used to compute **state values (V)** and **action values (Q)** and determine the **optimal path** in a simple environment.

Using **Value Iteration** and **Q-Learning**, the rover successfully learns the strategy that maximizes cumulative reward and reaches the most beneficial goal state.
